# Import the libraries and configure a spark session

In [ ]:
from pyspark.sql import SparkSession
import os
import numpy as np
import pandas as pd
from pyspark.sql.functions import pandas_udf
import pyspark.sql.functions as sf

In [ ]:
# Get the access and secret keys to connect to s3
access_key = os.getenv("S3_ACCESS_KEY")
secret_key = os.getenv("S3_SECRET_KEY") 

# Create a spark session including modules to connect to s3
spark = SparkSession.builder \
    .master("spark://master:7077")\
    .appName("Anomaly Detection")\
    .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.4.1,org.apache.hadoop:hadoop-common:3.4.1')\
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
    .config("spark.sql.execution.arrow.pyspark.fallback.enabled", "false")\
    .config('spark.hadoop.fs.s3a.aws.credentials.proviAnvoder', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')\
    .config('spark.hadoop.fs.s3a.access.key', access_key)\
    .config('spark.hadoop.fs.s3a.secret.key', secret_key)\
    .config('spark.hadoop.fs.s3a.endpoint', 'https://cloud-areapd.pd.infn.it:5210')\
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.metadatastore.impl", "org.apache.hadoop.fs.s3a.s3guard.NullMetadataStore") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled","false") \
    .config("com.amazonaws.sdk.disableCertChecking","true") \
    .getOrCreate()

# Task 1: conversion of alarms

First of all, we will convert the "A5" and "A9" variables from their integer representation to their bit string one. This will help identifying the required alarms. \
First, we have to read the data file from the CloudVeneto "bucket", converted into the *Parquet* format by  `df_spark.write.mode("overwrite").parquet("s3a://MAPDB-Group5/data_parquet")`

In [ ]:
# Read data from the CloudVeneto "bucket" through s3
df_spark = spark.read.parquet("s3a://MAPDB-Group5/data_parquet")
df_spark.show(10)

After reading the data, we need to normalize everything to the same sampling frequency before actually conducting the analysis.

In [ ]:
# Overwrite the "when" column casting that to the native "TimestampType"
# In order to do that, we need to divide for 1000 to get rid of milliseconds
df_spark = df_spark.withColumn(
    "when",
    (df_spark.when / 1000).cast("timestamp")
)

# Now let's add a "timestamp_bucket" column, "truncating" everything to the minutes
# to get the same sampling frequency for all data
df_spark = df_spark.withColumn(
    "timestamp_bucket",
    sf.date_trunc("minute", "when")
)

# Use flag "truncate=False" to show the whole timestamp
df_spark.show(10, truncate=False)

In [ ]:
# Converting the A5 and A9 metrics to their bit-string representation
# Note: all the other values are converted into strings because Spark
# wants the same type returned by when/otherwise 
df_spark = df_spark.withColumn(
    "BitString",
    sf.when(
        df_spark.metric.isin('A5', 'A9'),
        sf.lpad(sf.bin(df_spark.value), 16, '0')
    ).otherwise(df_spark.value.cast('string'))
)

Now that we have converted the 'A5' and 'A9' variables to their bit-string representation, we can identify the requested alarms by checking if 1+ bit(s) in position 6, 7 and 8 (staring from the LSB), are 1 in either or both of them, which means that engines are overheating.

In [ ]:
df_spark = df_spark.withColumn(
    "is_overheated",
    sf.when(
        df_spark.metric.isin('A5', 'A9') & 
        (sf.substring(df_spark.BitString, 6, 3) != '000'),
        True
    ).otherwise(False)
)

In [ ]:
df_spark.createOrReplaceTempView("prova")

spark.sql(
    """
    SELECT * FROM prova
    WHERE is_overheated = true
    """
).show(10)